Red convolucional

In [2]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader
from torch import nn, optim
import kagglehub
import os
import pandas as pd
from PIL import Image
import torch.nn.functional as F

C:\Users\Daniel\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import os
import torch
import kagglehub
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms


# --- 1. Configuración y Descarga de Kaggle ---
KAGGLE_DATASET_ID = 'pkdarabi/cardetection'
print(f"Descargando dataset: {KAGGLE_DATASET_ID}...")
KAGGLE_DOWNLOAD_PATH = kagglehub.dataset_download(KAGGLE_DATASET_ID)
print(f"Dataset descargado en: {KAGGLE_DOWNLOAD_PATH}")


# --- CÓDIGO PARA GENERAR EL DATASET ---
IMAGE_SIZE = (416, 416) 
LABELS_NAME = 'yolo_labels_dataset.csv'

# --- 4. Transformaciones ---
transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE), # Asegúrate que coincida con el cálculo del modelo
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # Normalizar
])


# --- 5. Clase YOLODataset (Modificada para aceptar un DataFrame) ---
class YOLODataset(Dataset):
    def __init__(self, labels_df, directorio_imagenes, transform=None):
        self.full_labels_df = labels_df
        self.directorio_imagenes = directorio_imagenes
        self.transform = transform
        self.imagenes_unicas = self.full_labels_df['nombre_archivo'].unique()
        self.labels_grouped = self.full_labels_df.groupby('nombre_archivo')

    def __len__(self):
        return len(self.imagenes_unicas)

    def __getitem__(self, idx):
        image_name = self.imagenes_unicas[idx]
        image_path = os.path.join(self.directorio_imagenes, image_name)
        
        try:
            image = Image.open(image_path).convert('RGB')
        except FileNotFoundError:
            print(f"Advertencia: No se encontró la imagen {image_path}.")
            # Devolver algo para que no se rompa el lote
            return torch.zeros(3, IMAGE_SIZE[0], IMAGE_SIZE[1]), torch.tensor(-1) 

        boxes_df = self.labels_grouped.get_group(image_name)
        
        # Tomar la primera etiqueta (modo Clasificación)
        clase_idx = int(boxes_df['clase_indice'].iloc[0])
        label = torch.tensor(clase_idx, dtype=torch.long)

        if self.transform:
            image = self.transform(image)

        return image, label


# 2. Definir las rutas usando la ruta de descarga de Kaggle
ruta_csv = os.path.join("./", LABELS_NAME) # El CSV se creó en el directorio actual
# IMPORTANTE: Definir la ruta de imágenes APUNTANDO al subdirectorio 'train/images'
ruta_imgs = os.path.join(KAGGLE_DOWNLOAD_PATH, 'car', 'train', 'images') 

# Opcional: Guardar el modelo entrenado
torch.save(model.state_dict(), 'yolo_classifier_model.pth')
print("Modelo guardado en 'yolo_classifier_model.pth'")

Descargando dataset: pkdarabi/cardetection...
Dataset descargado en: C:\Users\ivanp\.cache\kagglehub\datasets\pkdarabi\cardetection\versions\5
Usando dispositivo: cpu

Procesando el archivo CSV: yolo_labels_dataset.csv
Filas originales: 4279
Clases originales: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13)]
Filas después del procesado: 4012
Clases re-mapeadas: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]
Procesamiento del CSV completado.


NameError: name 'train_test_split' is not defined

In [4]:
# --- Configuración del Loader ---
BATCH_SIZE = 16 # Tamaño de batch comúnmente usado en detección de objetos.

# Crear el DataLoader
train_loader = DataLoader(dataset_yolo, batch_size=BATCH_SIZE, shuffle=True)
print(f"\nTipo de DataLoader creado: {type(train_loader)}")

print(f"\nDataLoader creado con éxito. Número de batches: {len(train_loader)}")


Tipo de DataLoader creado: <class 'torch.utils.data.dataloader.DataLoader'>

DataLoader creado con éxito. Número de batches: 220


In [5]:
class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet, self).__init__()
        # Primera capa convolutiva
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)  # Max pooling de 2x2
        # Segunda capa convolutiva
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        # Capa completamente conectada
        self.fc1 = nn.Linear(64 * 104 * 104, 120)  # Ajustamos correctamente las dimensiones
        self.fc2 = nn.Linear(120, 15)  # 15 categorías de salida
        self.softmax = nn.Softmax(dim=1)  # Función softmax para la capa de salida

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # Conv1 -> ReLU -> Pooling
        x = self.pool(F.relu(self.conv2(x)))  # Conv2 -> ReLU -> Pooling
        x = x.view(-1, 64 * 104 * 104)  # Aplanamos el tensor
        x = F.relu(self.fc1(x))  # FC1 -> ReLU
        x = self.fc2(x)  # FC2
        x = self.softmax(x)
        return x

In [ ]:
# Instanciamos la red y configuramos el entrenamiento
model = ConvNet()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Entrenamiento
epochs = 40
for epoch in range(epochs):
    running_loss = 0.0
    for images, labels in train_loader:  # Asegúrate de que `train_loader` esté correctamente definido
        optimizer.zero_grad()  # Limpiamos los gradientes
        outputs = model(images)  # Pasamos las imágenes por la red
        labels_one_hot = F.one_hot(labels, 15).float()
        loss = criterion(outputs, labels_one_hot)  # Calculamos la pérdida
        loss.backward()  # Backpropagation
        optimizer.step()  # Actualizamos los pesos

        running_loss += loss.item()
    print(f'Epoch {epoch+1}, Loss: {running_loss / len(train_loader)}')

Epoch 1, Loss: 2.7445674891924967


KeyboardInterrupt: 

In [ ]:
# --- Configuración de rutas para test ---
IMAGEN_DIR_TEST = os.path.join(KAGGLE_DOWNLOAD_PATH, 'car', 'test', 'images') 
ETIQUETAS_DIR_TEST = os.path.join(KAGGLE_DOWNLOAD_PATH, 'car', 'test', 'labels')
CSV_SALIDA_TEST = 'yolo_labels_test.csv'

# Definir las rutas usando la ruta de descarga de Kaggle
ruta_csv_test = os.path.join(DATA_DIR, CSV_SALIDA_TEST) # El CSV se creó en el directorio actual
# IMPORTANTE: Definir la ruta de imágenes APUNTANDO al subdirectorio 'train/images'
ruta_imgs_test = os.path.join(KAGGLE_DOWNLOAD_PATH, 'car', 'test', 'images') 

# Crear DataLoader de test
dataset_yolo_test = YOLODataset(archivo_csv=ruta_csv_test, directorio_imagenes=ruta_imgs_test, transform=transform)
test_loader = DataLoader(dataset_yolo_test, batch_size=BATCH_SIZE, shuffle=False)

print(f"\nTest DataLoader creado con {len(test_loader)} batches")



Test DataLoader creado con 40 batches


In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Evaluación de la red
def evaluate(model, test_loader):
    model.eval()  # Poner el modelo en modo evaluación
    correct = 0
    total = test_loader.dataset.__len__()  # Total de muestras en el conjunto de test
    print(f'Total de muestras en el conjunto de test: {total}')
    with torch.no_grad():  # No calcular gradientes
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)  # Mover datos al dispositivo
            outputs = model(inputs)  # Forward pass
            _, predicted = torch.max(outputs.data, 1)  # Obtener las predicciones
            correct += (predicted == labels).sum().item()  # Actualizar el contador de aciertos
    accuracy = 100 * correct / total if total > 0 else 0
    print(f'Accuracy: {accuracy:.2f}%')

In [ ]:
evaluate(model, test_loader)

Total de muestras en el conjunto de test: 637
Accuracy: 51.65%
